<a href="https://colab.research.google.com/github/Iamzain804/SentinelShield-AI---Weapon-Detection-System/blob/main/Gun_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Ye code automatically dataset download karega
import kagglehub
path = kagglehub.dataset_download("kruthisb999/guns-and-knifes-detection-in-cctv-videos")

100%|██████████| 989M/989M [00:10<00:00, 102MB/s]

Extracting files...


In [2]:
!pip install ultralytics
from ultralytics import YOLO
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
import os

# Find data.yaml file
for root, dirs, files in os.walk(path):
    if 'data.yaml' in files:
        data_yaml_path = os.path.join(root, 'data.yaml')
        print(f"Found data.yaml at: {data_yaml_path}")
        break

# Check dataset structure
!ls -la {path}

Found data.yaml at: /root/.cache/kagglehub/datasets/kruthisb999/guns-and-knifes-detection-in-cctv-videos/versions/1/combined_gunsnknifes/data.yaml
total 12
drwxr-xr-x 3 root root 4096 Mar 16 03:49 .
drwxr-xr-x 3 root root 4096 Mar 16 03:49 ..
drwxr-xr-x 5 root root 4096 Mar 16 03:49 combined_gunsnknifes


In [5]:
# Load YOLOv8 nano model (fastest)
model = YOLO('yolov8n.pt')

print("Model loaded successfully!")

Model loaded successfully!


In [6]:
results = model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    lr0=0.01,
    patience=20,
    optimizer="AdamW",
    name="gun_knife_detection",
    device=0
)

Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/.cache/kagglehub/datasets/kruthisb999/guns-and-knifes-detection-in-cctv-videos/versions/1/combined_gunsnknifes/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=gun_knife_detection

In [7]:
# Validate the model
metrics = model.val()

print(f"mAP50: {metrics.box.map50}")
print(f"mAP50-95: {metrics.box.map}")

Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1118.4±493.0 MB/s, size: 43.0 KB)
val: Scanning /root/.cache/kagglehub/datasets/kruthisb999/guns-and-knifes-detection-in-cctv-videos/versions/1/combined_gunsnknifes/val/labels.cache... 324 images, 17 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 324/324 84.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 3.9it/s 5.4s
                   all        324        339      0.779      0.725      0.752      0.429
                pistol        190        222      0.778      0.613       0.68      0.361
                 knife        117        117       0.78      0.838      0.824      0.498
Speed: 2.3ms preprocess, 4.8ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to /content/runs/detect/val
mAP5

In [8]:
# Test on validation images
test_images_path = os.path.join(path, 'combined_gunsnknifes/val/images')

# Get first 5 test images
import glob
test_images = glob.glob(os.path.join(test_images_path, '*.jpg'))[:5]

# Run inference
results = model.predict(test_images, save=True, conf=0.5)

print("Predictions saved!")


0: 640x640 1 pistol, 19.9ms
1: 640x640 2 pistols, 19.9ms
2: 640x640 (no detections), 19.9ms
3: 640x640 (no detections), 19.9ms
4: 640x640 1 knife, 19.9ms
Speed: 8.8ms preprocess, 19.9ms inference, 2.4ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/predict
Predictions saved!


In [9]:
# Download trained model to your local machine
from google.colab import files

# Best model path
best_model_path = 'runs/detect/gun_knife_detection/weights/best.pt'

# Download
files.download(best_model_path)

print("Model downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model downloaded!


In [10]:
model.export(format='onnx')  # ONNX format
# model.export(format='tflite')  # TensorFlow Lite
# model.export(format='torchscript')  # TorchScript

print("Model exported!")

Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from '/content/runs/detect/gun_knife_detection/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 12 packages in 197ms
Prepared 4 packages in 6.94s
Installed 4 packages in 259ms
 + colorama==0.4.6
 + onnx==1.20.1
 + onnxruntime-gpu==1.24.3
 + onnxslim==0.1.88

requirements: AutoUpdate success ✅ 8.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.20.1 opset 20...
ONNX: slimming with onnxslim 0.1.88...
ONNX: export success ✅ 10.4s, saved as 